In [ ]:
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
st.set_page_config(page_title="Retail Demand & Inventory Dashboard",layout="wide")
st.title("Retail Demand Forecasting & Inventory Optimization")
st.markdown("""This dashboard helps retail decision-makers understand demand patterns, promotion impact, and inventory risks using data-driven insights.""")
def load_data():
    df = pd.read_csv("data/cleaned_BrightMart_retail_dataset.csv")
    inventory = pd.read_csv("data/inventory_policy.csv")
    return df, inventory

df, inventory_policy = load_data()
st.subheader(" Key Business KPIs")

col1, col2, col3, col4 = st.columns(4)

col1.metric("Total Revenue", f"${df['Revenue'].sum():,.0f}")
col2.metric("Units Sold", f"{df['Quantity'].sum():,.0f}")
col3.metric("Avg Order Value", f"${df['Revenue'].mean():.2f}")
col4.metric("Avg Promotion (%)", f"{df['Promotion'].mean():.1f}%")
st.subheader("📈 Demand Analysis")

selected_category = st.selectbox("Select Category",df['Category'].unique())
category_data = df[df['Category'] == selected_category]

fig, ax = plt.subplots()
category_data.groupby('Month')['Revenue'].sum().plot(ax=ax)
ax.set_title("Monthly Revenue Trend")
ax.set_ylabel("Revenue")
st.pyplot(fig)

fig, ax = plt.subplots()
category_data.boxplot(
    column='Revenue',
    by='Discount Applied',
    ax=ax
)
ax.set_title("Promotion vs Revenue")
ax.set_xlabel("Discount Applied")
ax.set_ylabel("Revenue")
st.pyplot(fig)


st.subheader("Inventory Optimization")

inv = inventory_policy[inventory_policy['Category'] == selected_category]
st.write("### Inventory Recommendation")
st.metric("Average Daily Demand", round(inv['Avg_Daily_Demand'].values[0], 2))
st.metric("Safety Stock", int(inv['Safety_Stock'].values[0]))
st.metric("Reorder Point", int(inv['Reorder_Point'].values[0]))

if inv['Stockout_Risk'].values[0]:
    st.error("High stockout risk during promotions")
else:
    st.success("Inventory sufficient for promotions")

st.subheader("Cost Impact Simulation")

col1, col2 = st.columns(2)

col1.metric(
    "Estimated Stockout Cost",
    f"${inv['Stockout_Cost'].values[0]:,.0f}"
)

col2.metric(
    "Holding Cost",
    f"${inv['Holding_Cost'].values[0]:,.0f}"
)
def load_feature_importance():
    return pd.read_csv("data/feature_importance.csv")

feature_importance = load_feature_importance()
st.subheader("What Drives Demand?")
top_features = (feature_importance.sort_values(by='Importance', ascending=False).head(10))
fig, ax = plt.subplots(figsize=(8, 5))

ax.barh(
    top_features['Feature'],
    top_features['Importance']
)

ax.set_xlabel("Importance Score")
ax.set_ylabel("Feature")
ax.set_title("Top 10 Demand Drivers")
ax.invert_yaxis()
ax.grid(axis='x', linestyle='--', alpha=0.6)

st.pyplot(fig)
st.markdown("""
### Key Insights
- **Stock availability** is the strongest driver of realized demand
- **Price sensitivity** plays a major role in purchase decisions
- **Promotions increase demand only when inventory is sufficient**
- **Category-level behavior varies**, requiring tailored inventory strategies

### Business Takeaway
Discounts do not create sales if products are unavailable.
Inventory readiness is more critical than promotion intensity.
""")
